# Experiment H1: Causal Antecedent Intervention Analysis

**Research Question:** Is there an antecedent trigger token preceding object hallucinations in LLaVA-1.5-7B under teacher-forcing causal intervention?

**Core Objectives:**
1. Intervene at upstream positions $j \in [t-10, t-2]$ by substituting replacement candidates $c$ with balanced split-half assignment (A/B).
2. Test whether hallucinated objects differ from 1:1 same-category matched real objects across endpoints:
   - **E0:** Structure existence (1-sample Wilcoxon on $\log HPR$ vs 0 + permutation null)
   - **E1:** Magnitude disparity ($\Delta \log Mbar = \log(Mbar_h / Mbar_r)$)
   - **E2:** Concentration disparity ($\Delta \log HPR = \log HPR_h - \log HPR_r$)
   - **E3:** Profile reproducibility ($\Delta \rho_z = \rho_{z,h} - \rho_{z,r}$)
3. Evaluate Decision Table outcomes (O1–O5), local syntactic dominance (O5: $d=2$), and Sensitivities S1–S5.

**Prerequisites:**
- Dual Tesla T4 GPUs (`device_map="auto"`).
- Input: Experiment 1 notebook output containing `labels.jsonl` (and `lag_records.csv` if available).
- Input: `biminhco/val2014` COCO images.

In [ ]:
# CELL 1: Environment setup and GPU verification
import subprocess, sys, os

print("=== GPU Status ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

print("Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.37.0", "accelerate>=0.26.0",
     "bitsandbytes", "sentencepiece", "tqdm", "scipy", "scikit-learn",
     "spacy>=3.7.0", "pandas", "matplotlib"],
    check=True
)

print("Downloading spaCy en_core_web_sm pipeline...")
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
print("Environment setup complete!")

In [ ]:
# CELL 2: Clone or update repository & define real-time unbuffered runner
import subprocess, sys, os

REPO = "/kaggle/working/Experiment_CTI"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/ntmy12/Experiment_CTI.git", REPO], check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "main"], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO, check=True)

def run_streaming(cmd, cwd=REPO):
    """Executes a command with real-time, unbuffered stdout/stderr streaming (tqdm friendly)."""
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    p = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in iter(p.stdout.readline, ""):
        sys.stdout.write(line)
        sys.stdout.flush()
    p.stdout.close()
    ret = p.wait()
    if ret != 0:
        raise RuntimeError(f"Command failed with exit code {ret}: {' '.join(cmd)}")

print("Repository ready at:", REPO)
print("Files in src/:", os.listdir(os.path.join(REPO, "src")))


In [ ]:
# CELL 3: Gate A Verification -- Offline Unit Tests (U1 - U6)
print("Executing Gate A unit tests in real time...")
run_streaming([sys.executable, "-u", "-m", "unittest", "tests/test_h1_units.py"])
print("[GATE A PASSED] All offline unit tests verified.
")


In [ ]:
# CELL 4: Locate datasets (labels.jsonl, lag_records.csv, and COCO images)
import os, glob, shutil

REPO = "/kaggle/working/Experiment_CTI"

# 1. labels.jsonl
labels_cands = glob.glob("/kaggle/input/**/labels.jsonl", recursive=True)
if not labels_cands:
    raise FileNotFoundError("labels.jsonl not found in /kaggle/input/. Add Experiment 1 notebook output.")
os.makedirs(os.path.join(REPO, "data"), exist_ok=True)
shutil.copy(labels_cands[0], os.path.join(REPO, "data", "labels.jsonl"))
n_lines = sum(1 for _ in open(os.path.join(REPO, "data", "labels.jsonl")))
print(f"Found labels.jsonl ({n_lines} captions) at: {labels_cands[0]}")

# 2. lag_records.csv (optional, for Gate B G4)
lag_cands = glob.glob("/kaggle/input/**/lag_records.csv", recursive=True)
lag_records_path = lag_cands[0] if lag_cands else ""
print(f"Exp 1 lag_records: {lag_records_path if lag_records_path else 'None (G4 will be skipped)'}")

# 3. COCO images
sample = glob.glob("/kaggle/input/**/COCO_val2014_*.jpg", recursive=True)
if not sample:
    raise FileNotFoundError("COCO val2014 images not found. Add biminhco/val2014 dataset.")
IMAGE_DIR = os.path.dirname(sample[0])
print(f"COCO images found at: {IMAGE_DIR} ({len(glob.glob(os.path.join(IMAGE_DIR, '*.jpg')))} images)")

In [ ]:
# CELL 5: Step 1 -- Pair Construction (07_make_pairs_h1.py)
import json, os

print("Constructing 1:1 same-category matched pairs...")
run_streaming([
    sys.executable, "-u", "src/07_make_pairs_h1.py",
    "--labels_file", "data/labels.jsonl",
    "--output_dir",  "results/h1",
    "--caliper",     "0.10",
    "--K",           "10",
    "--seed",        "0",
])

funnel_path = os.path.join(REPO, "results/h1/funnel_pairs.json")
if os.path.exists(funnel_path):
    with open(funnel_path) as f:
        funnel = json.load(f)
    print("\nPair Funnel Summary:")
    print(json.dumps(funnel, indent=2))
assert os.path.exists(os.path.join(REPO, "results/h1/pairs_h1.csv")), "pairs_h1.csv missing!"


In [ ]:
# CELL 6: Gate B Verification -- GPU Control Tests (G0 - G5 on 20 Objects)
print("Executing Gate B GPU control tests in real time...")
cmd = [
    sys.executable, "-u", "src/08_run_h1.py",
    "--labels_file",      "data/labels.jsonl",
    "--pairs_file",       "results/h1/pairs_h1.csv",
    "--synonyms_file",    "data/synonyms.txt",
    "--image_dir",        IMAGE_DIR,
    "--output_dir",       "results/h1",
    "--run_controls",
]
if lag_records_path:
    cmd.extend(["--lag_records_path", lag_records_path])

run_streaming(cmd)
print("[GATE B PASSED] All GPU control tests satisfied.
")


In [ ]:
# CELL 7: Gate C & D -- Pilot 100 Pairs & Freeze Configuration
import time, json, os, hashlib

print("Running Pilot 100 pairs to benchmark wall-clock throughput...")
t0 = time.time()
run_streaming([
    sys.executable, "-u", "src/08_run_h1.py",
    "--labels_file",   "data/labels.jsonl",
    "--pairs_file",    "results/h1/pairs_h1.csv",
    "--synonyms_file", "data/synonyms.txt",
    "--image_dir",     IMAGE_DIR,
    "--output_dir",    "results/h1",
    "--limit",         "100",
    "--batch_size",    "8",
])
elapsed = time.time() - t0
sec_per_pair = elapsed / 100.0
print(f"Pilot 100 pairs completed in {elapsed:.1f}s ({sec_per_pair:.2f}s/pair).")

# Freeze Configuration
def hash_file(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()

frozen_cfg = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO).decode().strip(),
    "sha256_07_make_pairs": hash_file(os.path.join(REPO, "src/07_make_pairs_h1.py")),
    "sha256_08_run": hash_file(os.path.join(REPO, "src/08_run_h1.py")),
    "sha256_09_analyze": hash_file(os.path.join(REPO, "src/09_analyze_h1.py")),
    "sha256_h1_common": hash_file(os.path.join(REPO, "src/h1_common.py")),
    "K": 10,
    "caliper": 0.10,
    "seed": 0,
    "batch_size": 8,
    "pos_mode": "spacy",
}
cfg_path = os.path.join(REPO, "results/h1/config_frozen.json")
with open(cfg_path, "w") as f:
    json.dump(frozen_cfg, f, indent=2)
print("\nFrozen config created at:", cfg_path)
print(json.dumps(frozen_cfg, indent=2))


In [ ]:
# CELL 8: Gate E -- Full Confirmation Cohort Execution
print("Starting full confirmation cohort causal intervention run...")
run_streaming([
    sys.executable, "-u", "src/08_run_h1.py",
    "--labels_file",          "data/labels.jsonl",
    "--pairs_file",           "results/h1/pairs_h1.csv",
    "--synonyms_file",        "data/synonyms.txt",
    "--image_dir",            IMAGE_DIR,
    "--output_dir",           "results/h1",
    "--batch_size",           "8",
    "--checkpoint_interval",  "25",
])
print("Full causal intervention run finished successfully!
")


In [ ]:
# CELL 9: Statistical Analysis & Reporting (09_analyze_h1.py)
print("Evaluating endpoints E0 - E3, Decision Table O1 - O5, Sensitivities S1 - S5...")
run_streaming([
    sys.executable, "-u", "src/09_analyze_h1.py",
    "--objects_file",    "results/h1/h1_objects.csv",
    "--candidates_file", "results/h1/h1_candidates.csv",
    "--pairs_file",      "results/h1/pairs_h1.csv",
    "--labels_file",     "data/labels.jsonl",
    "--output_dir",      "results/h1",
    "--n_boot",          "5000",
    "--seed",            "0",
])
print("Analysis and report generation complete!
")


In [ ]:
# CELL 10: Display REPORT.md and Figures
import os
from IPython.display import display, Markdown, Image

REPO = "/kaggle/working/Experiment_CTI"
report_path = os.path.join(REPO, "results/h1/REPORT.md")
if os.path.exists(report_path):
    with open(report_path) as f:
        display(Markdown(f.read()))

fig1 = os.path.join(REPO, "results/h1/figures/distance_profile.png")
fig2 = os.path.join(REPO, "results/h1/figures/logHPR_by_group.png")
if os.path.exists(fig1):
    display(Image(fig1))
if os.path.exists(fig2):
    display(Image(fig2))